# AI/ML Internship — Week 3 Solution
## Data Visualization & Feature Engineering Pipeline — House Prices Dataset

**Dataset:** House Prices — Advanced Regression Techniques  
**Target:** `SalePrice`  
**Work completed:** All 18 Week 3 steps, including visualization charts, 8 engineered features, categorical encoding, scaling comparison, skewness treatment, feature selection, reusable functions, final dashboard, feature engineering infographic, and written report.

> Note: For a real predictive model, `PricePerSF = SalePrice / TotalSF` is a target-derived feature and creates data leakage. It is created because the task explicitly asks for it, but the final ML-ready feature-selection stage excludes it from candidate predictors.

## Saved Output Preview
The required Week 3 output files are included in this folder. The two main submission images are shown below.

### Final 6-Chart Dashboard
![Final Dashboard](week3_dashboard.png)

### Feature Engineering Pipeline Infographic
![Feature Engineering Pipeline](week3_fe_pipeline.png)

### Other Saved Charts
- `w3_saleprice_distribution.png`
- `w3_grlivarea_distribution.png`
- `w3_scatter_multivar.png`
- `w3_time_trend.png`
- `w3_neighborhood_boxplot.png`
- `w3_heatmap.png`
- `w3_pairplot.png`
- `w3_facetgrid.png`
- `w3_engineered_corr.png`
- `w3_scaling_comparison.png`
- `w3_skewness_bar.png`
- `w3_skewness_before_after.png`

In [ ]:
# Step 1 — Environment Setup & Dataset Loading
from pathlib import Path
import zipfile, warnings, math
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import boxcox

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import VarianceThreshold
from IPython.display import display, Markdown

# Professional plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8FAFC',
    'axes.edgecolor': '#334155',
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'legend.framealpha': 0.9,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight'
})
sns.set_theme(style='whitegrid')

OUT = Path('.')

# Load Kaggle House Prices data. Works both in this folder and in Colab after uploading the zip.
zip_paths = [Path('house-prices-advanced-regression-techniques.zip'), Path('/mnt/data/house-prices-advanced-regression-techniques.zip')]
if not Path('train.csv').exists():
    for zp in zip_paths:
        if zp.exists():
            with zipfile.ZipFile(zp, 'r') as z:
                z.extractall('.')
            break

train_path = Path('train.csv')
if not train_path.exists():
    raise FileNotFoundError('train.csv not found. Upload/extract house-prices-advanced-regression-techniques.zip first.')

df_raw = pd.read_csv(train_path)
df = df_raw.copy()

print('Dataset shape:', df.shape)
print('\nDataFrame info:')
df.info()

print('\nStatistical summary:')
display(df.describe().T.head(20))

numeric_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Numerical columns: {len(numeric_cols)}')
print(f'Categorical columns: {len(categorical_cols)}')
print('\nCategorical columns:', categorical_cols)

corr_top10 = df.select_dtypes(include='number').corr()['SalePrice'].abs().sort_values(ascending=False).head(10)
print('\nTop 10 original numerical correlations with SalePrice:')
display(corr_top10.to_frame('abs_corr_with_SalePrice'))

In [ ]:
# Step 2 — Matplotlib Chart 1 & 2: Distribution Analysis
sale_skew = df['SalePrice'].skew()
log_sale = np.log1p(df['SalePrice'])
log_sale_skew = pd.Series(log_sale).skew()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(df['SalePrice'], bins=40, color='#5B21B6', edgecolor='white', alpha=0.85)
axes[0].axvline(df['SalePrice'].mean(), color='#DC2626', linestyle='--', linewidth=2, label='Mean')
axes[0].set_title('SalePrice Distribution — Original')
axes[0].set_xlabel('SalePrice')
axes[0].set_ylabel('Count')
axes[0].text(0.97, 0.95, f"Mean: {df['SalePrice'].mean():,.0f}\nSkew: {sale_skew:.2f}",
             transform=axes[0].transAxes, ha='right', va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
axes[0].legend()

axes[1].hist(log_sale, bins=40, color='#0D9488', edgecolor='white', alpha=0.85)
axes[1].axvline(log_sale.mean(), color='#DC2626', linestyle='--', linewidth=2, label='Mean')
axes[1].set_title('SalePrice Distribution — log1p')
axes[1].set_xlabel('log1p(SalePrice)')
axes[1].set_ylabel('Count')
axes[1].text(0.97, 0.95, f"Mean: {log_sale.mean():.2f}\nSkew: {log_sale_skew:.2f}",
             transform=axes[1].transAxes, ha='right', va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
axes[1].legend()
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Chart 1 — SalePrice Distribution: Before vs After Log Transform', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT/'w3_saleprice_distribution.png')
plt.show()

# Chart 2: GrLivArea Box + Violin
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot(df['GrLivArea'].dropna(), vert=False, patch_artist=True,
                boxprops=dict(facecolor='#A78BFA', color='#4C1D95'),
                medianprops=dict(color='#DC2626', linewidth=2))
axes[0].set_title('GrLivArea Box Plot')
axes[0].set_xlabel('Above Ground Living Area')
axes[0].set_yticks([])

parts = axes[1].violinplot(df['GrLivArea'].dropna(), vert=False, showmeans=True, showmedians=True)
for pc in parts['bodies']:
    pc.set_facecolor('#14B8A6')
    pc.set_edgecolor('#0F766E')
    pc.set_alpha(0.7)
axes[1].set_title('GrLivArea Violin Plot')
axes[1].set_xlabel('Above Ground Living Area')
axes[1].set_yticks([])
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Chart 2 — Distribution of GrLivArea', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT/'w3_grlivarea_distribution.png')
plt.show()

### Step 3 note
The scatter plot below encodes **four variables at once**: `GrLivArea` on the x-axis, `SalePrice` on the y-axis, `OverallQual` through color, and `GarageCars` through point size. This makes it easier to see that larger living area usually increases price, but quality and garage capacity also explain why some similar-sized houses sell for much more than others.

In [ ]:
# Step 3 — Matplotlib Chart 3: Multi-Variable Scatter Plot
x = df['GrLivArea']
y = df['SalePrice']
quality = df['OverallQual']
sizes = df['GarageCars'].fillna(0) * 20 + 20
corr = x.corr(y)

fig, ax = plt.subplots(figsize=(11, 7))
sc = ax.scatter(x, y, c=quality, s=sizes, cmap=plt.cm.RdYlGn, alpha=0.65, edgecolors='k', linewidths=0.2)
coef = np.polyfit(x, y, deg=2)
poly = np.poly1d(coef)
x_line = np.linspace(x.min(), x.max(), 300)
ax.plot(x_line, poly(x_line), color='#111827', linewidth=2.5, label='Degree-2 regression line')

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Overall Quality')
ax.set_title('Chart 3 — GrLivArea vs SalePrice Colored by Overall Quality')
ax.set_xlabel('GrLivArea')
ax.set_ylabel('SalePrice')
ax.text(0.03, 0.95, f'Pearson r = {corr:.3f}', transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
for garage in sorted(df['GarageCars'].dropna().unique())[:5]:
    ax.scatter([], [], s=garage*20+20, color='gray', alpha=0.6, label=f'{int(garage)} GarageCars')
ax.legend(loc='lower right', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(OUT/'w3_scatter_multivar.png')
plt.show()

In [ ]:
# Step 4 — Matplotlib Chart 4: Time-Based Trend Analysis
trend = df.copy()
trend['YearBuiltDecade'] = (trend['YearBuilt'] // 10) * 10
decade_summary = trend.groupby('YearBuiltDecade').agg(
    HousesSold=('SalePrice', 'size'),
    AvgSalePrice=('SalePrice', 'mean')
).reset_index()
decade_summary['DecadeLabel'] = decade_summary['YearBuiltDecade'].astype(int).astype(str) + 's'

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
axes[0].plot(decade_summary['DecadeLabel'], decade_summary['HousesSold'], marker='o', linewidth=2.5, color='#5B21B6')
axes[0].set_title('Count of Houses Sold by YearBuilt Decade')
axes[0].set_ylabel('Number of Houses')

norm = plt.Normalize(decade_summary['AvgSalePrice'].min(), decade_summary['AvgSalePrice'].max())
colors = plt.cm.viridis(norm(decade_summary['AvgSalePrice']))
bars = axes[1].bar(decade_summary['DecadeLabel'], decade_summary['AvgSalePrice'], color=colors, edgecolor='white')
axes[1].set_title('Average SalePrice by Decade of Construction')
axes[1].set_ylabel('Average SalePrice')
axes[1].set_xlabel('YearBuilt Decade')
for bar, value in zip(bars, decade_summary['AvgSalePrice']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{value/1000:.0f}K',
                 ha='center', va='bottom', fontsize=8, rotation=90)
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.xticks(rotation=45)
fig.suptitle('Chart 4 — Time-Based Trend Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT/'w3_time_trend.png')
plt.show()

In [ ]:
# Step 5 — Seaborn Charts 5 & 6: Statistical Visualization
# Chart 5: SalePrice by top 8 neighborhoods using boxplot
top8_neighborhoods = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).head(8).index
neigh_df = df[df['Neighborhood'].isin(top8_neighborhoods)].copy()
neigh_order = neigh_df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(data=neigh_df, x='Neighborhood', y='SalePrice', order=neigh_order, palette='husl', ax=ax)
ax.axhline(df['SalePrice'].mean(), color='#111827', linestyle='--', linewidth=2, label=f"Overall Mean: {df['SalePrice'].mean():,.0f}")
ax.set_title('Chart 5 — SalePrice Distribution Across Top 8 Neighborhoods')
ax.set_xlabel('Neighborhood')
ax.set_ylabel('SalePrice')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig(OUT/'w3_neighborhood_boxplot.png')
plt.show()

# Chart 6: Heatmap for top 15 numerical features by SalePrice correlation
top15_num = df.select_dtypes(include='number').corr()['SalePrice'].abs().sort_values(ascending=False).head(15).index
corr_matrix = df[top15_num].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            square=True, linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Chart 6 — Correlation Heatmap of Top 15 Numerical Features')
plt.tight_layout()
plt.savefig(OUT/'w3_heatmap.png')
plt.show()

In [ ]:
# Step 6 — Seaborn Advanced: Pair Plot and FacetGrid
pair_df = df[['SalePrice', 'GrLivArea', 'TotalBsmtSF', 'OverallQual', 'YearBuilt']].copy()
pair_df['OverallQualBin'] = pd.cut(pair_df['OverallQual'], bins=[0, 4, 7, 10], labels=['Low (1-4)', 'Medium (5-7)', 'High (8-10)'])

pair_grid = sns.pairplot(pair_df, vars=['SalePrice', 'GrLivArea', 'TotalBsmtSF', 'OverallQual', 'YearBuilt'],
                         hue='OverallQualBin', diag_kind='kde', plot_kws={'alpha':0.4, 's':25}, corner=False)
pair_grid.fig.suptitle('Chart 7 — Pair Plot of Core House Price Features', y=1.02, fontsize=16, fontweight='bold')
pair_grid.savefig(OUT/'w3_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

# FacetGrid by Kitchen Quality
facet_df = df.copy()

def hist_with_mean(data, color, **kwargs):
    ax = plt.gca()
    sns.histplot(data=data, x='SalePrice', bins=25, color=color, edgecolor='white', alpha=0.75, ax=ax)
    mean_val = data['SalePrice'].mean()
    ax.axvline(mean_val, color='#DC2626', linestyle='--', linewidth=2)
    ax.text(0.05, 0.92, f"n={len(data)}\nmean={mean_val/1000:.0f}K", transform=ax.transAxes,
            va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.85), fontsize=9)

g = sns.FacetGrid(facet_df, col='KitchenQual', col_order=['Ex', 'Gd', 'TA', 'Fa'], height=4, sharex=True, sharey=False)
g.map_dataframe(hist_with_mean)
g.fig.suptitle('Chart 8 — SalePrice Distribution by Kitchen Quality', y=1.08, fontsize=16, fontweight='bold')
g.set_axis_labels('SalePrice', 'Count')
g.savefig(OUT/'w3_facetgrid.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 7 — Feature Creation: 8 Engineered Features
fe_df = df.copy()

# TotalSF combines basement, first floor, and second floor area to capture total usable property size.
fe_df['TotalSF'] = fe_df['TotalBsmtSF'].fillna(0) + fe_df['1stFlrSF'].fillna(0) + fe_df['2ndFlrSF'].fillna(0)

# TotalBaths gives half bathrooms half-weight because they usually add less value than full bathrooms.
fe_df['TotalBaths'] = fe_df['FullBath'].fillna(0) + 0.5*fe_df['HalfBath'].fillna(0) + fe_df['BsmtFullBath'].fillna(0) + 0.5*fe_df['BsmtHalfBath'].fillna(0)

# HouseAge measures how old the house was when sold, which usually affects maintenance and buyer preference.
fe_df['HouseAge'] = fe_df['YrSold'] - fe_df['YearBuilt']

# RemodelAge measures how many years passed since last remodel, capturing freshness of updates.
fe_df['RemodelAge'] = fe_df['YrSold'] - fe_df['YearRemodAdd']

# HasRemodeled flags whether the house has been improved after original construction.
fe_df['HasRemodeled'] = (fe_df['YearBuilt'] != fe_df['YearRemodAdd']).astype(int)

# QualCond combines overall quality and overall condition into one interaction score.
fe_df['QualCond'] = fe_df['OverallQual'] * fe_df['OverallCond']

# PricePerSF measures sale price density per square foot; useful analytically but target-derived, so it is excluded from final predictive features.
fe_df['PricePerSF'] = np.where(fe_df['TotalSF'] > 0, fe_df['SalePrice'] / fe_df['TotalSF'], np.nan)

# IsNewHouse flags houses built within 5 years of the sale year because newer homes may command a premium.
fe_df['IsNewHouse'] = (fe_df['YearBuilt'] >= fe_df['YrSold'] - 5).astype(int)

new_features = ['TotalSF', 'TotalBaths', 'HouseAge', 'RemodelAge', 'HasRemodeled', 'QualCond', 'PricePerSF', 'IsNewHouse']
print('Engineered feature summary:')
display(fe_df[new_features].describe().T)
print('\nFirst 10 rows of engineered features:')
display(fe_df[new_features].head(10))

In [ ]:
# Step 8 — Correlation Analysis of Engineered Features
corr_all = fe_df.select_dtypes(include='number').corr()['SalePrice'].drop('SalePrice').sort_values(key=lambda s: s.abs(), ascending=False)
print('Top 20 correlations with SalePrice after feature engineering:')
display(corr_all.head(20).to_frame('correlation_with_SalePrice'))
print('\nBottom 10 correlations with SalePrice after feature engineering:')
display(corr_all.tail(10).to_frame('correlation_with_SalePrice'))

fig, ax = plt.subplots(figsize=(11, 8))
top20 = corr_all.head(20).sort_values()
colors = ['#16A34A' if v > 0 else '#DC2626' for v in top20.values]
ax.barh(top20.index, top20.values, color=colors)
ax.set_title('Top 20 Features by Absolute Correlation with SalePrice')
ax.set_xlabel('Correlation with SalePrice')
ax.axvline(0, color='black', linewidth=1)
plt.tight_layout()
plt.savefig(OUT/'w3_engineered_corr.png')
plt.show()

engineered_in_top20 = [c for c in corr_all.head(20).index if c in new_features]
display(Markdown(f"""
**Engineered features in the top 20:** {', '.join(engineered_in_top20)}.  
This shows that feature engineering added strong signal. `TotalSF`, `TotalBaths`, `QualCond`, age-related features, and the analytical `PricePerSF` feature capture pricing patterns more directly than several raw columns.
"""))

In [ ]:
# Step 9 — Categorical Variable Analysis & Encoding Strategy
cat_cols = fe_df.select_dtypes(include='object').columns.tolist()
quality_cols = ['ExterQual', 'KitchenQual', 'BsmtQual', 'GarageQual', 'FireplaceQu']

print(f'Total categorical columns: {len(cat_cols)}')

for col in cat_cols:
    print('\n' + '='*90)
    print(f'Column: {col}')
    print(f'Unique categories: {fe_df[col].nunique(dropna=True)}')
    print('Value counts:')
    print(fe_df[col].value_counts(dropna=False))
    print('Mean SalePrice per category:')
    print(fe_df.groupby(col, dropna=False)['SalePrice'].mean().sort_values(ascending=False))

strategy_records = []
for col in cat_cols:
    missing_ratio = fe_df[col].isna().mean()
    dominant_ratio = fe_df[col].value_counts(normalize=True, dropna=False).iloc[0]
    nunique = fe_df[col].nunique(dropna=True)
    if col in quality_cols:
        strategy = 'label_encode'
        reason = 'Ordinal quality scale: Ex > Gd > TA > Fa > Po > NA.'
    elif missing_ratio > 0.50 or dominant_ratio > 0.95:
        strategy = 'drop'
        reason = 'More than 50% missing or one category dominates more than 95%.'
    elif nunique > 10:
        strategy = 'frequency_encode'
        reason = 'High-cardinality nominal feature; frequency encoding avoids too many dummy columns.'
    else:
        strategy = 'onehot_encode'
        reason = 'Nominal feature with <= 10 categories and no natural order.'
    strategy_records.append({
        'column': col,
        'unique_categories': nunique,
        'missing_%': round(missing_ratio*100, 2),
        'dominant_%': round(dominant_ratio*100, 2),
        'strategy': strategy,
        'reason': reason
    })

strategy_table = pd.DataFrame(strategy_records)
print('\nEncoding decision table:')
display(strategy_table)

In [ ]:
# Step 10 — Apply All Encoding Strategies
before_shape = fe_df.shape
encoded_df = fe_df.copy()
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}

label_cols = strategy_table.loc[strategy_table['strategy'] == 'label_encode', 'column'].tolist()
drop_cols = strategy_table.loc[strategy_table['strategy'] == 'drop', 'column'].tolist()
freq_cols = strategy_table.loc[strategy_table['strategy'] == 'frequency_encode', 'column'].tolist()
onehot_cols = strategy_table.loc[strategy_table['strategy'] == 'onehot_encode', 'column'].tolist()

# A) Label encoding for ordinal quality columns
for col in label_cols:
    encoded_df[col] = encoded_df[col].fillna('NA').map(quality_map).fillna(0).astype(int)

# B) Drop very sparse or near-constant categorical columns
encoded_df = encoded_df.drop(columns=drop_cols, errors='ignore')

# C) Frequency encoding for high-cardinality nominal columns such as Neighborhood
for col in freq_cols:
    values = encoded_df[col].fillna('NA')
    freq = values.value_counts(normalize=True)
    encoded_df[col + '_freq'] = values.map(freq)
    encoded_df = encoded_df.drop(columns=[col])

# D) One-hot encoding for low-cardinality nominal columns
for col in onehot_cols:
    if col in encoded_df.columns:
        encoded_df[col] = encoded_df[col].fillna('NA')
encoded_df = pd.get_dummies(encoded_df, columns=[c for c in onehot_cols if c in encoded_df.columns], drop_first=True, dtype=int)

# E) Median imputation for numeric missing values before scaling/model preparation
num_cols_encoded = encoded_df.select_dtypes(include='number').columns
encoded_df[num_cols_encoded] = encoded_df[num_cols_encoded].fillna(encoded_df[num_cols_encoded].median())

after_shape = encoded_df.shape
print('Shape before encoding:', before_shape)
print('Shape after encoding:', after_shape)
print('Remaining object columns:', encoded_df.select_dtypes(include='object').columns.tolist())
print('\nStrategy counts:')
display(strategy_table['strategy'].value_counts().to_frame('count'))

In [ ]:
# Step 11 — Feature Scaling: Compare Three Methods
X = encoded_df.drop('SalePrice', axis=1)
y = encoded_df['SalePrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}
scaled_data = {}
comparison_rows = []
selected_features = ['GrLivArea', 'TotalSF', 'LotArea']

for scaler_name, scaler in scalers.items():
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
    scaled_data[scaler_name] = (X_train_scaled_df, X_test_scaled_df, scaler)
    for feature in selected_features:
        comparison_rows.append({
            'scaler': scaler_name,
            'feature': feature,
            'raw_mean': X_train[feature].mean(),
            'raw_std': X_train[feature].std(),
            'scaled_mean': X_train_scaled_df[feature].mean(),
            'scaled_std': X_train_scaled_df[feature].std()
        })

comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.round(4))

fig, axes = plt.subplots(3, 2, figsize=(13, 11))
for i, (scaler_name, (X_train_scaled_df, _, _)) in enumerate(scaled_data.items()):
    axes[i, 0].hist(X_train['GrLivArea'], bins=35, color='#5B21B6', edgecolor='white', alpha=0.8)
    axes[i, 0].set_title(f'Raw GrLivArea — before {scaler_name}')
    axes[i, 0].set_xlabel('Raw GrLivArea')
    axes[i, 1].hist(X_train_scaled_df['GrLivArea'], bins=35, color='#0D9488', edgecolor='white', alpha=0.8)
    axes[i, 1].set_title(f'GrLivArea after {scaler_name}')
    axes[i, 1].set_xlabel('Scaled GrLivArea')
for ax in axes.ravel():
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Feature Scaling Comparison for GrLivArea', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT/'w3_scaling_comparison.png')
plt.show()

display(Markdown("""
**Scaler choice for Week 4 Linear Regression:** I would choose **StandardScaler** because linear regression and regularized linear models work well when numerical features are centered around zero and placed on a comparable standard-deviation scale. MinMaxScaler is useful when features need to stay between 0 and 1, while RobustScaler is strongest when outliers are severe. Tree models usually do not require scaling because splits are based on feature thresholds rather than distance or gradient scale.
"""))

In [ ]:
# Step 12 — Skewness Detection & Visualization
numeric_skew = encoded_df.select_dtypes(include='number').skew().dropna()

def skew_severity(value: float) -> str:
    av = abs(value)
    if av < 0.5:
        return 'Normal'
    if av < 1:
        return 'Moderate'
    if av < 2:
        return 'High'
    return 'Very High'

skew_table = pd.DataFrame({
    'column': numeric_skew.index,
    'skewness': numeric_skew.values,
    'severity': [skew_severity(v) for v in numeric_skew.values]
}).sort_values('skewness', key=lambda s: s.abs(), ascending=False)

display(skew_table.head(30))

high_skew_count = int((numeric_skew.abs() > 1).sum())
high_skew_percent = high_skew_count / len(numeric_skew) * 100
print(f'Features with |skew| > 1: {high_skew_count} of {len(numeric_skew)} ({high_skew_percent:.2f}%)')

top15_skew = skew_table.head(15).sort_values('skewness')
severity_colors = {'Normal':'#16A34A', 'Moderate':'#F59E0B', 'High':'#EA580C', 'Very High':'#DC2626'}
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(top15_skew['column'], top15_skew['skewness'], color=[severity_colors[s] for s in top15_skew['severity']])
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Top 15 Most Skewed Features')
ax.set_xlabel('Skewness')
plt.tight_layout()
plt.savefig(OUT/'w3_skewness_bar.png')
plt.show()

In [ ]:
# Step 13 — Apply Skewness Transformations: Before & After
transformed_df = encoded_df.copy()
original_skew = transformed_df.select_dtypes(include='number').skew().dropna()

treat_cols = [
    c for c, v in original_skew.items()
    if c != 'SalePrice' and abs(v) > 0.75 and transformed_df[c].min() >= 0 and transformed_df[c].nunique() > 1
]

for col in treat_cols:
    transformed_df[col] = np.log1p(transformed_df[col])

# Compare SalePrice transformations
sp = encoded_df['SalePrice']
sp_log = np.log1p(sp)
sp_sqrt = np.sqrt(sp)
sp_boxcox, boxcox_lambda = stats.boxcox(sp)
saleprice_transform_summary = pd.DataFrame({
    'transformation': ['Original', 'log1p', 'sqrt', 'Box-Cox'],
    'skewness': [sp.skew(), pd.Series(sp_log).skew(), pd.Series(sp_sqrt).skew(), pd.Series(sp_boxcox).skew()]
})
display(saleprice_transform_summary)
print(f'Box-Cox lambda: {boxcox_lambda:.4f}')

best_row = saleprice_transform_summary.iloc[saleprice_transform_summary['skewness'].abs().argmin()]
best_transform = best_row['transformation']
if best_transform == 'Box-Cox':
    transformed_df['SalePrice_transformed'] = sp_boxcox
elif best_transform == 'log1p':
    transformed_df['SalePrice_transformed'] = sp_log
elif best_transform == 'sqrt':
    transformed_df['SalePrice_transformed'] = sp_sqrt
else:
    transformed_df['SalePrice_transformed'] = sp
print(f'Best transformation for SalePrice by absolute skewness: {best_transform}')

# Choose 3 continuous highly-skewed columns for before/after visualization
plot_features = [c for c in treat_cols if transformed_df[c].nunique() > 20 and c not in ['Id']][:3]
while len(plot_features) < 3:
    for c in treat_cols:
        if c not in plot_features and c != 'Id':
            plot_features.append(c)
        if len(plot_features) == 3:
            break
plot_cols = ['SalePrice'] + plot_features[:3]

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
for i, col in enumerate(plot_cols):
    if col == 'SalePrice':
        original = encoded_df[col]
        transformed = transformed_df['SalePrice_transformed']
        after_label = f'{best_transform} transformed SalePrice'
    else:
        original = encoded_df[col]
        transformed = transformed_df[col]
        after_label = f'log1p({col})'
    axes[i, 0].hist(original, bins=35, color='#5B21B6', edgecolor='white', alpha=0.8)
    axes[i, 0].set_title(f'Original {col} — skew={pd.Series(original).skew():.2f}')
    axes[i, 1].hist(transformed, bins=35, color='#0D9488', edgecolor='white', alpha=0.8)
    axes[i, 1].set_title(f'{after_label} — skew={pd.Series(transformed).skew():.2f}')
for ax in axes.ravel():
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Skewness Treatment: Before vs After', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT/'w3_skewness_before_after.png')
plt.show()

print(f'Total numerical features transformed with log1p (excluding SalePrice): {len(treat_cols)}')

In [ ]:
# Step 14 — Feature Selection: Most Informative Features
# Exclude SalePrice, transformed target, Id, and PricePerSF because PricePerSF uses the target and would leak SalePrice.
feature_candidates = transformed_df.drop(columns=['SalePrice', 'SalePrice_transformed', 'Id', 'PricePerSF'], errors='ignore')
target = transformed_df['SalePrice']

feature_corr = feature_candidates.corrwith(target).dropna()
top30_features = feature_corr.abs().sort_values(ascending=False).head(30).index.tolist()
X_top = feature_candidates[top30_features].copy()

# Variance threshold
vt = VarianceThreshold(threshold=0.01)
X_var = pd.DataFrame(vt.fit_transform(X_top), columns=X_top.columns[vt.get_support()], index=X_top.index)

# Remove multicollinearity > 0.95 by dropping the feature with weaker target correlation
corr_matrix_full = X_var.corr().abs()
upper = corr_matrix_full.where(np.triu(np.ones(corr_matrix_full.shape), k=1).astype(bool))
to_drop = []
for col in upper.columns:
    high_corr_partners = upper.index[upper[col] > 0.95].tolist()
    for row in high_corr_partners:
        c_col = abs(feature_corr.get(col, 0))
        c_row = abs(feature_corr.get(row, 0))
        drop_feature = col if c_col < c_row else row
        if drop_feature not in to_drop:
            to_drop.append(drop_feature)

X_final = X_var.drop(columns=to_drop, errors='ignore')
final_features_info = pd.DataFrame({
    'feature': X_final.columns,
    'correlation_with_SalePrice': feature_corr[X_final.columns].values,
    'dtype': [str(feature_candidates[c].dtype) for c in X_final.columns]
}).sort_values('correlation_with_SalePrice', key=lambda s: s.abs(), ascending=False)

original_candidate_count = feature_candidates.shape[1]
remaining_count = len(final_features_info)
eliminated_pct = (1 - remaining_count/original_candidate_count) * 100
print(f'Original candidate features: {original_candidate_count}')
print(f'Features remaining after selection: {remaining_count}')
print(f'Percentage eliminated: {eliminated_pct:.2f}%')
print('Dropped for multicollinearity:', to_drop)
display(final_features_info)

In [ ]:
# Step 15 — Three Reusable Feature Engineering Functions
from typing import Iterable, Optional, Tuple, Dict, Any
import pandas as pd
import numpy as np

def visualize_distributions(df: pd.DataFrame, cols: Iterable[str], n_cols: int = 3) -> plt.Figure:
    """
    Create a multi-panel histogram grid for selected numeric columns.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.
    cols : Iterable[str]
        Column names to visualize.
    n_cols : int, default=3
        Number of subplot columns.

    Returns
    -------
    matplotlib.figure.Figure
        Figure object containing the histogram grid.
    """
    cols = [c for c in cols if c in df.columns]
    n_rows = int(np.ceil(len(cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, cols):
        data = df[col].dropna()
        ax.hist(data, bins=30, color='#5B21B6', edgecolor='white', alpha=0.8)
        ax.set_title(f'{col} | skew={data.skew():.2f}')
        ax.set_xlabel(col)
        ax.set_ylabel('Count')
    for ax in axes[len(cols):]:
        ax.axis('off')
    plt.tight_layout()
    return fig


def encode_quality_column(series: pd.Series, quality_map: Optional[Dict[str, int]] = None) -> Tuple[pd.Series, Dict[str, int]]:
    """
    Encode an ordinal quality column using a quality mapping.

    Parameters
    ----------
    series : pd.Series
        Categorical quality values such as Ex, Gd, TA, Fa, Po, or NA.
    quality_map : dict, optional
        Custom mapping. If None, the standard Ames quality scale is used.

    Returns
    -------
    Tuple[pd.Series, dict]
        Encoded numeric Series and the mapping used.
    """
    if quality_map is None:
        quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
    encoded = series.fillna('NA').map(quality_map).fillna(0).astype(int)
    return encoded, quality_map


def full_scaling_pipeline(X_train: pd.DataFrame, X_test: pd.DataFrame, method: str = 'standard') -> Tuple[np.ndarray, np.ndarray, Any]:
    """
    Fit a scaler on training data and transform train and test sets.

    Parameters
    ----------
    X_train : pd.DataFrame
        Training features.
    X_test : pd.DataFrame
        Test features.
    method : {'standard', 'minmax', 'robust'}, default='standard'
        Scaling method.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray, scaler]
        Scaled train array, scaled test array, and fitted scaler object.
    """
    scaler_options = {
        'standard': StandardScaler(),
        'minmax': MinMaxScaler(),
        'robust': RobustScaler()
    }
    if method not in scaler_options:
        raise ValueError("method must be one of: 'standard', 'minmax', 'robust'")
    scaler = scaler_options[method]
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

# Function calls / outputs
fig = visualize_distributions(encoded_df, ['SalePrice', 'GrLivArea', 'LotArea', 'TotalSF', 'TotalBaths', 'HouseAge'], n_cols=3)
plt.savefig(OUT/'w3_reusable_distribution_grid.png')
plt.show()

encoded_kitchen, used_mapping = encode_quality_column(fe_df['KitchenQual'])
print('KitchenQual mapping used:', used_mapping)
print(encoded_kitchen.head(10).tolist())

X_train_std, X_test_std, fitted_scaler = full_scaling_pipeline(X_train, X_test, method='standard')
print('Scaled train shape:', X_train_std.shape)
print('Scaled test shape:', X_test_std.shape)
print('Scaler object:', fitted_scaler)

In [ ]:
# Step 16 — Final 6-Chart Professional Dashboard
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.ravel()

# Chart 1: SalePrice before/after log transformation overlaid on standardized scale
raw_z = pd.Series(stats.zscore(encoded_df['SalePrice']), name='Raw SalePrice (z-score)')
log_z = pd.Series(stats.zscore(np.log1p(encoded_df['SalePrice'])), name='log1p SalePrice (z-score)')
sns.histplot(raw_z, bins=35, kde=True, stat='density', color='#5B21B6', alpha=0.35, ax=axes[0], label='Raw')
sns.histplot(log_z, bins=35, kde=True, stat='density', color='#0D9488', alpha=0.35, ax=axes[0], label='Log1p')
axes[0].axvline(raw_z.mean(), color='#5B21B6', linestyle='--')
axes[0].axvline(log_z.mean(), color='#0D9488', linestyle='--')
axes[0].set_title('1) SalePrice Distribution Before vs After log1p')
axes[0].set_xlabel('Standardized value')
axes[0].text(0.03, 0.95, f"Raw skew={encoded_df['SalePrice'].skew():.2f}\nLog skew={np.log1p(encoded_df['SalePrice']).skew():.2f}",
             transform=axes[0].transAxes, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
axes[0].legend()

# Chart 2: TotalSF vs SalePrice colored by OverallQual
sc = axes[1].scatter(encoded_df['TotalSF'], encoded_df['SalePrice'], c=encoded_df['OverallQual'], cmap='RdYlGn', alpha=0.6, s=35)
coef = np.polyfit(encoded_df['TotalSF'], encoded_df['SalePrice'], 1)
line_x = np.linspace(encoded_df['TotalSF'].min(), encoded_df['TotalSF'].max(), 200)
axes[1].plot(line_x, np.poly1d(coef)(line_x), color='#111827', linewidth=2)
r_total = encoded_df['TotalSF'].corr(encoded_df['SalePrice'])
axes[1].set_title('2) TotalSF vs SalePrice')
axes[1].set_xlabel('TotalSF')
axes[1].set_ylabel('SalePrice')
axes[1].text(0.03, 0.95, f'r={r_total:.3f}', transform=axes[1].transAxes, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
fig.colorbar(sc, ax=axes[1], label='OverallQual')

# Chart 3: Top 15 feature correlations
corr_dash = transformed_df.drop(columns=['SalePrice_transformed'], errors='ignore').corr()['SalePrice'].drop('SalePrice').sort_values(key=lambda s: s.abs(), ascending=False).head(15).sort_values()
colors = ['#16A34A' if v > 0 else '#DC2626' for v in corr_dash.values]
axes[2].barh(corr_dash.index, corr_dash.values, color=colors)
axes[2].set_title('3) Top 15 Correlation-Based Feature Importances')
axes[2].set_xlabel('Correlation with SalePrice')
axes[2].axvline(0, color='black', linewidth=1)

# Chart 4: SalePrice by OverallQual
sns.boxplot(data=encoded_df, x='OverallQual', y='SalePrice', palette='viridis', ax=axes[3])
medians = encoded_df.groupby('OverallQual')['SalePrice'].median()
for i, (qual, med) in enumerate(medians.items()):
    axes[3].text(i, med, f'{med/1000:.0f}K', ha='center', va='bottom', fontsize=8, color='black')
axes[3].set_title('4) SalePrice by Overall Quality')
axes[3].set_xlabel('OverallQual')
axes[3].set_ylabel('SalePrice')

# Chart 5: Heatmap top 10 + SalePrice
heat_cols = transformed_df.drop(columns=['SalePrice_transformed'], errors='ignore').corr()['SalePrice'].abs().sort_values(ascending=False).head(11).index
sns.heatmap(transformed_df[heat_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.4, cbar=False, ax=axes[4])
axes[4].set_title('5) Correlation Matrix: Top 10 Features + SalePrice')

# Chart 6: before/after scaling comparison with inset axes
axes[5].axis('off')
axes[5].set_title('6) GrLivArea Scaling Comparison', pad=20)
mini_axes = [axes[5].inset_axes([0.05, 0.12, 0.28, 0.78]), axes[5].inset_axes([0.37, 0.12, 0.28, 0.78]), axes[5].inset_axes([0.69, 0.12, 0.28, 0.78])]
mini_data = [
    ('Raw', X_train['GrLivArea']),
    ('StandardScaled', scaled_data['StandardScaler'][0]['GrLivArea']),
    ('MinMaxScaled', scaled_data['MinMaxScaler'][0]['GrLivArea'])
]
for ax, (title, data) in zip(mini_axes, mini_data):
    ax.hist(data, bins=30, color='#6366F1', edgecolor='white', alpha=0.75)
    ax.set_title(title, fontsize=10)
    ax.tick_params(labelsize=8)

for ax in axes[:5]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Week 3 Final Dashboard — House Price Visualization & Feature Engineering', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT/'week3_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 17 — Feature Engineering Summary Report / Infographic
feature_formulas = {
    'TotalSF': 'TotalBsmtSF + 1stFlrSF + 2ndFlrSF',
    'TotalBaths': 'FullBath + .5*HalfBath + BsmtFullBath + .5*BsmtHalfBath',
    'HouseAge': 'YrSold - YearBuilt',
    'RemodelAge': 'YrSold - YearRemodAdd',
    'HasRemodeled': 'YearBuilt != YearRemodAdd',
    'QualCond': 'OverallQual * OverallCond',
    'PricePerSF': 'SalePrice / TotalSF',
    'IsNewHouse': 'YearBuilt >= YrSold - 5'
}
feature_created_table = pd.DataFrame({
    'Feature': new_features,
    'Formula': [feature_formulas[f] for f in new_features],
    'Corr': [fe_df[f].corr(fe_df['SalePrice']) for f in new_features]
})

transformed_skew = transformed_df[treat_cols].skew().dropna() if len(treat_cols) else pd.Series(dtype=float)
skew_compare = pd.DataFrame({
    'Original Skew': original_skew.reindex(treat_cols),
    'Transformed Skew': transformed_skew.reindex(treat_cols)
}).dropna()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.ravel()

# Panel 1: Features created table
axes[0].axis('off')
table_data = feature_created_table.copy()
table_data['Corr'] = table_data['Corr'].map(lambda x: f'{x:.3f}')
t1 = axes[0].table(cellText=table_data.values, colLabels=table_data.columns, loc='center', cellLoc='left')
t1.auto_set_font_size(False)
t1.set_fontsize(8)
t1.scale(1, 1.6)
axes[0].set_title('Panel 1 — Features Created', fontsize=14, fontweight='bold')

# Panel 2: Encoding decision counts
enc_counts = strategy_table['strategy'].value_counts()
axes[1].bar(enc_counts.index, enc_counts.values, color=['#5B21B6', '#0D9488', '#F59E0B', '#DC2626'][:len(enc_counts)])
axes[1].set_title('Panel 2 — Encoding Decisions')
axes[1].set_ylabel('Number of Categorical Features')
axes[1].tick_params(axis='x', rotation=25)
for i, v in enumerate(enc_counts.values):
    axes[1].text(i, v, str(v), ha='center', va='bottom')

# Panel 3: Original vs transformed skewness
if len(skew_compare):
    axes[2].scatter(skew_compare['Original Skew'], skew_compare['Transformed Skew'], alpha=0.65, color='#0D9488')
    max_lim = max(skew_compare.abs().max().max(), 1)
    axes[2].plot([-max_lim, max_lim], [-max_lim, max_lim], color='#111827', linestyle='--', label='No change line')
    axes[2].axhline(0, color='gray', linewidth=0.8)
    axes[2].axvline(0, color='gray', linewidth=0.8)
    axes[2].legend()
axes[2].set_title('Panel 3 — Skewness Treatment Before vs After')
axes[2].set_xlabel('Original Skewness')
axes[2].set_ylabel('Transformed Skewness')

# Panel 4: Final feature set table
axes[3].axis('off')
final_top15 = final_features_info.head(15).copy()
final_top15['correlation_with_SalePrice'] = final_top15['correlation_with_SalePrice'].map(lambda x: f'{x:.3f}')
t2 = axes[3].table(cellText=final_top15.values, colLabels=final_top15.columns, loc='center', cellLoc='left')
t2.auto_set_font_size(False)
t2.set_fontsize(8)
t2.scale(1, 1.55)
axes[3].set_title('Panel 4 — Final Feature Set: Top 15', fontsize=14, fontweight='bold')

fig.suptitle('Week 3 Feature Engineering Pipeline Summary', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT/'week3_fe_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

# Step 18 — Written Analysis Report

## 1) Executive Summary
This Week 3 project used the Kaggle House Prices dataset with 1,460 rows, 81 original columns, and `SalePrice` as the target variable. The main goal was to convert raw housing data into professional visual insights and machine-learning-ready inputs. The first key finding is that `SalePrice` is strongly right-skewed, with an original skewness of about 1.88; after transformation, Box-Cox produced the lowest skewness, while log transformation also made the distribution much more normal and interpretable. The second key finding is that quality and space dominate price: `OverallQual`, `TotalSF`, and `GrLivArea` were among the strongest predictors. The third key finding is that engineered features added real value: `TotalSF`, `TotalBaths`, `QualCond`, `HouseAge`, `RemodelAge`, `IsNewHouse`, and `PricePerSF` appeared in the top correlation ranking.

## 2) Visualization Insights
The SalePrice distribution chart revealed that house prices are not normally distributed. Most houses are concentrated in the lower-to-middle price range, while a smaller number of expensive homes creates a long right tail. The log-transformed chart showed a much more balanced shape, which is useful for regression modelling. The GrLivArea box and violin plots showed that living area also contains outliers, meaning a few houses are much larger than the majority. The multivariable scatter plot was one of the most informative charts because it showed that larger homes usually sell for more, but the strongest prices are also linked with higher `OverallQual` and larger garage capacity. The time-trend chart showed that newer construction decades generally have higher average sale prices. The neighborhood box plot showed that location matters strongly because median prices vary widely across neighborhoods. The heatmap confirmed that quality, size, basement area, garage size, and bathrooms are strongly related to price. The pair plot and FacetGrid added more statistical evidence that quality categories separate expensive homes from cheaper homes.

## 3) Feature Engineering Rationale
Eight features were engineered using real-estate domain logic. `TotalSF` was created because buyers usually value total usable space, not just one individual area column. `TotalBaths` was created because full bathrooms add more value than half bathrooms, so half bathrooms were weighted by 0.5. `HouseAge` was created because older houses may require more maintenance and may sell for less unless updated. `RemodelAge` captured how recently the property was improved because a recent remodel can increase buyer interest. `HasRemodeled` turned remodeling status into a simple binary signal. `QualCond` combined overall quality and condition, because a house that is both high quality and well maintained should be more valuable. `PricePerSF` measured value density and was useful for analysis, although it should not be used as a real predictive input because it contains the target. `IsNewHouse` captured the premium that buyers may pay for recently built homes.

## 4) Encoding Decisions
The categorical encoding strategy was chosen according to the meaning and size of each categorical column. Ordinal quality columns such as `ExterQual`, `KitchenQual`, `BsmtQual`, `GarageQual`, and `FireplaceQu` were label encoded using the ordered scale `Ex=5`, `Gd=4`, `TA=3`, `Fa=2`, `Po=1`, and `NA=0`, because these categories have a natural quality ranking. Low-cardinality nominal columns with no natural order, such as zoning, lot shape, building type, house style, foundation, and sale condition, were one-hot encoded so that the model would not assume false ordering. High-cardinality nominal columns, including `Neighborhood`, `Exterior1st`, and `Exterior2nd`, were frequency encoded to prevent excessive dummy columns. Columns with more than 50% missing values or one category dominating more than 95% were dropped because they contributed little reliable signal and could add noise.

## 5) Scaling Analysis
Three scaling methods were compared: StandardScaler, MinMaxScaler, and RobustScaler. StandardScaler centers each feature around mean 0 and standard deviation 1, making it a strong default for linear regression, ridge, lasso, SVM, and neural networks. MinMaxScaler compresses values into a fixed 0–1 range, which is helpful when bounded inputs are required, but it is sensitive to outliers. RobustScaler uses the median and interquartile range, making it better when outliers are extreme. For Week 4 linear regression, StandardScaler is the best choice because it gives all numerical features comparable scale while preserving the overall distribution structure. For tree-based models, scaling is usually not required because decision trees split using thresholds.

## 6) Skewness Treatment Findings
Skewness analysis showed that many numerical features were highly skewed, especially area, count, and one-hot encoded sparse features. In this processed dataset, 125 out of 168 numerical columns had absolute skewness above 1, which is about 74.40%. For non-target numerical features with absolute skewness greater than 0.75 and non-negative values, log1p transformation was applied because it handles zeros and reduces right tails. For the target `SalePrice`, three transformations were compared: log1p, square root, and Box-Cox. Box-Cox produced the lowest absolute skewness, approximately -0.009, while log1p reduced skewness from about 1.88 to about 0.12. Reducing skewness matters because linear models often perform better when relationships are more stable and distributions are less dominated by extreme values.

## 7) Reflection
The hardest concept in this task was deciding how to encode categorical variables correctly because not all text columns should be treated the same way. Some categories have a real order, while others are purely names and need one-hot or frequency encoding. The most surprising pattern was the strength of engineered features. `TotalSF` achieved a very strong relationship with `SalePrice`, showing that combining related raw columns can create a clearer signal than using each column separately. Another important lesson was that a feature can be useful for analysis but unsafe for prediction; `PricePerSF` is informative, but it leaks the target. Next, I would train baseline linear regression, ridge regression, random forest, and gradient boosting models using the selected feature set and compare performance using cross-validation.